# RSSM live diagnostics

Collect a short Crafter rollout, run the (still untrained) RSSM `observe` path,
and plot mechanism graphs **inline**: latent entropy, class occupancy, `h`
trajectory PCA, and imagination drift vs horizon.

Same helpers as `scripts/visualize_rssm.py` (`training.rssm_diagnostics`).
This is functionality you can re-run and poke — not a milestone checklist.


In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

from models.encoder import Encoder
from models.rssm import RSSM, one_hot_action, unimix_probs
from training.device import configure_runtime, describe_device, get_device, warn_if_not_cuda
from training.rollout import collect_sequences, encode_sequence
from training.rssm_diagnostics import (
    imagination_divergence,
    plot_entropy_over_time,
    plot_h_trajectory,
    plot_imagination_divergence,
    plot_latent_occupancy,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str((ROOT / ".mplcache").resolve()))

%matplotlib inline

CONFIG = Path("configs/m2_rssm.yaml")
with CONFIG.open() as f:
    cfg = yaml.safe_load(f)

seed = int(cfg["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = get_device()
configure_runtime(device)
print(f"device: {describe_device(device)}")
warn_if_not_cuda(device)


## Build encoder + RSSM

Random init until M3 trains the world model — plots below check mechanism health.


In [ ]:
enc_cfg = cfg["encoder"]
rssm_cfg = cfg["rssm"]
env_cfg = cfg["env"]
ver = cfg["verify"]
diag = cfg.get("diagnostics", {})
action_dim = int(env_cfg["action_dim"])

encoder = Encoder(
    embed_dim=int(enc_cfg["embed_dim"]),
    channels=tuple(int(c) for c in enc_cfg["channels"]),
).to(device)
rssm = RSSM(
    embed_dim=int(enc_cfg["embed_dim"]),
    action_dim=action_dim,
    deter_dim=int(rssm_cfg["deter_dim"]),
    stoch=int(rssm_cfg["stoch"]),
    classes=int(rssm_cfg["classes"]),
    hidden=int(rssm_cfg["hidden"]),
    unimix=float(rssm_cfg.get("unimix", 0.01)),
    act=str(rssm_cfg.get("act", "silu")),
    initial=str(rssm_cfg.get("initial", "learned")),
    rec_depth=int(rssm_cfg.get("rec_depth", 1)),
).to(device)
encoder.eval()
rssm.eval()
print(
    f"RSSM unimix={rssm.unimix} initial={rssm.initial_mode} "
    f"rec_depth={rssm.rec_depth} deter={rssm.deter_dim} z={rssm.stoch}x{rssm.classes}"
)


## Collect + observe

Uses a smaller batch/seq than the CLI script so the notebook stays snappy.
Bump these if you want denser plots.


In [ ]:
NUM_EPISODES = 2
SEQ_LEN = 48
BATCH = 2

obs_u8, actions_i = collect_sequences(
    env_id=str(env_cfg["id"]),
    num_episodes=NUM_EPISODES,
    seq_len=SEQ_LEN,
    max_episode_steps=int(ver["max_episode_steps"]),
    action_dim=action_dim,
)
obs_u8 = obs_u8[:BATCH]
actions_i = actions_i[:BATCH]

with torch.no_grad():
    embeds = encode_sequence(encoder, obs_u8, device)
    actions = one_hot_action(actions_i.to(device), action_dim)
    out = rssm.observe(embeds, actions)
    prior_probs = unimix_probs(out.prior_logits, rssm.unimix)
    posterior_probs = unimix_probs(out.posterior_logits, rssm.unimix)

print(
    f"obs {tuple(obs_u8.shape)}  embeds {tuple(embeds.shape)}  "
    f"h {tuple(out.h.shape)}  z_prior {tuple(out.z_prior.shape)}"
)

# Peek at the first frame of each sequence
fig, axes = plt.subplots(1, BATCH, figsize=(3 * BATCH, 3))
if BATCH == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow(obs_u8[i, 0].numpy())
    ax.set_title(f"seq {i}, t=0")
    ax.axis("off")
fig.suptitle("Grounded rollout starts")
fig.tight_layout()
plt.show()


## Latent entropy over time


In [ ]:
fig, prior_ent, post_ent = plot_entropy_over_time(prior_probs, posterior_probs)
plt.show()
print(
    f"mean entropy — prior {prior_ent.mean():.3f}, posterior {post_ent.mean():.3f} "
    f"(max uniform = {np.log(rssm.classes):.3f})"
)


## Class occupancy heatmaps

Near-uniform / "underused" is expected on a random init. After M3 training this
should specialize; stuck-near-uniform would flag collapse.


In [ ]:
fig, dead_prior = plot_latent_occupancy(prior_probs, "z_prior class occupancy")
plt.show()
fig, dead_post = plot_latent_occupancy(posterior_probs, "z_posterior class occupancy")
plt.show()
print(f"underused-class fraction — prior {dead_prior:.3f}, posterior {dead_post:.3f}")


## Deterministic state trajectory (PCA)


In [ ]:
fig = plot_h_trajectory(out.h)
plt.show()


## Imagination drift vs horizon

Open-loop `img_step` from several starts, compared to grounded `h` that kept
seeing real observations. Expect drift to grow; the *rate* matters once trained.


In [ ]:
horizon = int(diag.get("imagination_horizon", min(24, embeds.shape[1] - 1)))
num_starts = int(diag.get("imagination_num_starts", 4))
mean_curve, std_curve = imagination_divergence(
    rssm, out, actions, horizon=horizon, num_starts=num_starts
)
fig = plot_imagination_divergence(mean_curve, std_curve)
plt.show()
print(
    f"drift step1={mean_curve[0]:.3f}  step{len(mean_curve)}={mean_curve[-1]:.3f}"
)


## Optional: save current figures to `results/m2/`

Uncomment if you want durable PNGs from this run (same paths as the CLI script).


In [ ]:
SAVE = False  # flip to True to write PNGs

if SAVE:
    out_dir = Path(diag.get("results_dir", "results/m2"))
    fig, _, _ = plot_entropy_over_time(
        prior_probs, posterior_probs, out_dir / "latent_entropy.png"
    )
    plt.close(fig)
    fig, _ = plot_latent_occupancy(
        prior_probs, "z_prior class occupancy", out_dir / "latent_occupancy_prior.png"
    )
    plt.close(fig)
    fig, _ = plot_latent_occupancy(
        posterior_probs,
        "z_posterior class occupancy",
        out_dir / "latent_occupancy_posterior.png",
    )
    plt.close(fig)
    fig = plot_h_trajectory(out.h, out_dir / "h_trajectory_pca.png")
    plt.close(fig)
    fig = plot_imagination_divergence(
        mean_curve, std_curve, out_dir / "imagination_divergence.png"
    )
    plt.close(fig)
    print(f"wrote PNGs under {out_dir}/")
else:
    print("SAVE=False — figures shown inline only")
